**Predictions of CT, predictions of coliform bacteria and Khom  trained model and graphs**

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import joblib
import pandas as pd
from sklearn.preprocessing import StandardScaler
from scipy.optimize import curve_fit
import torch
import torch.nn as nn
import torch.optim as optim

def filter_by_datetime(colprint, start_date, start_hour, end_date, end_hour,chambnumb):
    # Ensure Date column is datetime
    colprint['Date'] = pd.to_datetime(colprint['Date'])

    # Create full datetime column (combine Date + hour)
    colprint['DateTime'] = colprint['Date']# + pd.to_timedelta(ozoneprint['hour'], unit='h')

    # Build start and end datetime
    start_dt = pd.to_datetime(start_date)# + pd.to_timedelta(start_hour, unit='h')
    end_dt = pd.to_datetime(end_date)# + pd.to_timedelta(end_hour, unit='h')

    mask = (
        (colprint['DateTime'] >= start_dt) &
        (colprint['DateTime'] <= end_dt) &
        (colprint['hour'] >= start_hour) &
        (colprint['hour'] <= end_hour) &
        (colprint['Chamber'] == chambnumb)
    )
    return colprint[mask].copy()

def colbactplots(colprint):
    # --- Step 1: Select dates ---
    start_date=input("Set the start date in the format of 2024-10-01):")
    #start_date="2024-10-01"
    start_hour=int(input("Set the start hour (0-23):"))
    #start_hour=12
    end_date=input("Set the end date in the format of 2024-10-01):")
    #end_date="2024-10-20"
    end_hour=int(input("Set the end hour (0-23):"))
    #end_hour=22
    chambnumb=int(input("Set the chamber (1-4):"))
    filtered_df = filter_by_datetime(colprint, start_date, start_hour, end_date, end_hour,chambnumb)

    if filtered_df.empty:
        print("No data in the specified date/hour range.")
        return filtered_df,None

    scaler_features = ['Chamber','WaterTemperature','RFTurbidity','FlowRate','Ozonedosage', 'RFturbidity_1', 'CT_All']
    X_full = filtered_df[scaler_features].copy()

    # --- Step 2: Load ensemble model and predict results ---
    save_dir = "/content/drive/My Drive/Colab Notebooks/Ozonation model/Coliform Bacteria best model/"
    modelname="ada_logcol_7_42.pkl"
    featurescaler_path = os.path.join(save_dir, "feature_scaler.pkl")
    targetscaler_path = os.path.join(save_dir, "target_scaler.pkl")
    model_path = os.path.join(save_dir, modelname)
    #xgb_model = joblib.load(model_path)
    #feature_scaler = joblib.load(featurescaler_path)
    #target_scaler = joblib.load(targetscaler_path)

    # --- Step 3: Load Model and Scalers ---
    try:
       ada_model = joblib.load(model_path)
       feature_scaler = joblib.load(featurescaler_path)
       target_scaler = joblib.load(targetscaler_path)
       print("✅ All models and scalers loaded successfully.")
    except FileNotFoundError as e:
        print(f"❌ File not found: {e}")
        return None, None
    except Exception as e:
        print(f"⚠️ Error loading: {e}")
        target_scaler = None # Fallback

    # --- Step 4: Base Case Prediction ---
    X_base = filtered_df[scaler_features].copy()
    y_pred_base_scaled = ada_model.predict(feature_scaler.transform(X_base))
    y_pred_base = target_scaler.inverse_transform(y_pred_base_scaled.reshape(-1, 1)).ravel()

    # --- Step 5: log removal predictions ---
    Clog_filtered = (X_base['Ozonedosage']- 0.012)/ np.log10(X_base['Ozonedosage'] / 0.012)
    time_filtered = filtered_df['Residencetime']
    log_rem_pred = piml_predict_log_removal(y_pred_base,Clog_filtered, time_filtered)

    # --- Step 5: Sensitivity Input ---
    print("\n--- Coliform Sensitivity Analysis ---")
    change_pct = float(input("Enter Ozonedosage change % (e.g., 10 or -10): "))
    multiplier = 1 + (change_pct / 100)

    # --- Step 6: Recalculate CT and Features ---
    # 1. Recalculate CT first
    ct_features = ['hour', 'month', 'Chamber', 'WaterTemperature', 'RFTurbidity', 'FlowRate', 'Ozonedosage','Residencetime']
    X_for_ct = filtered_df[ct_features].copy()
    X_for_ct['Ozonedosage'] = X_for_ct['Ozonedosage'] * multiplier
    new_ct = ctpredict(X_for_ct) # Assuming ctpredict is available in workspace

    # 2. Create perturbed feature set for XGBoost
    X_sens = X_base.copy()
    X_sens['Ozonedosage'] = X_sens['Ozonedosage'] * multiplier
    X_sens['CT_All'] = new_ct
    # If Ozonedosage was in your scaler_features list for this model, you'd update it here too

    # --- Step 6: Predict Sensitivity Case ---
    Clog_filtered_new = (X_for_ct['Ozonedosage']- 0.012)/ np.log10(X_for_ct['Ozonedosage'] / 0.012)
    y_pred_sens_scaled = ada_model.predict(feature_scaler.transform(X_sens))
    y_pred_sens = target_scaler.inverse_transform(y_pred_sens_scaled.reshape(-1, 1)).ravel()
    log_rem_sens = piml_predict_log_removal(y_pred_sens,Clog_filtered_new, time_filtered)

    # --- Step 7: Plot and compare results ---
    filtered_df['Log removal limit'] = 1.2
    plt.figure(figsize=(16,8))
    plt.plot(filtered_df['DateTime'], filtered_df['Log removal limit'], label="Minimum accepted removal rate",color='orange',linestyle=':', linewidth=2)
    plt.plot(filtered_df['DateTime'], log_rem_pred, label="Predicted Log Removal", marker='x',color='blue', linestyle='--')
    plt.scatter(filtered_df['DateTime'], -filtered_df['Coliform_log_removal'],color='red', s=100, zorder=5, label="Coliform Bacteria log removal calculated on measured inlet - outlet grab samples")
    plt.plot(filtered_df['DateTime'], log_rem_sens,
             label=f"Predicted Log Removal (Ozone Dosage {change_pct:+}%)",
             marker='x', color='green', linestyle='--')
    plt.xlabel("DateTime")
    plt.ylabel("Log Removal")
    plt.title(f"Coliform removal Rate Predictions vs Coliform Removal Rate in chamber{chambnumb}\n{start_date} {start_hour}:00 to {end_date} {end_hour}:00")
    plt.legend()
    plt.grid(True)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

    return filtered_df, log_rem_pred

def ctpredict(X_missing):

    save_dir = "/content/drive/My Drive/Colab Notebooks/Ozonation model/OZONE BEST TRAINED MODELS/New dataset/Mix features"
    #modelname="weightedensemble_model.pkl"
    modelname="GB_test_5_v2_42.pkl"
    featurescaler_path = os.path.join(save_dir, "feature_scaler.pkl")
    targetscaler_path = os.path.join(save_dir, "target_scaler.pkl")
    model_path = os.path.join(save_dir, modelname)
    #ensemble_model = joblib.load(model_path)
    gb_model = joblib.load(model_path)
    feature_scaler = joblib.load(featurescaler_path)
    target_scaler = joblib.load(targetscaler_path)
    #method = ensemble_model.get("method")
    #print("The ensemble model method is: ", method)
    model_features = ["Chamber", "WaterTemperature", "RFTurbidity", "FlowRate", "Ozonedosage"]

    #base_models = ensemble_model.get("models", [])
    #scaler_features = ["hour","month","Chamber","WaterTemperature","RFTurbidity","FlowRate","Ozonedosage","Residencetime"]

    def safe_predict(m, f, df):
        X_sub = df.reindex(columns=f, fill_value=0)
        scaler_features = feature_scaler.feature_names_in_ if hasattr(feature_scaler, "feature_names_in_") else f
        X_full = pd.DataFrame(0, index=df.index, columns=scaler_features)
        common_features = [c for c in scaler_features if c in X_sub.columns]
        X_full[common_features] = X_sub[common_features]
        X_scaled = feature_scaler.transform(X_full)
        model_feature_indices = [list(scaler_features).index(c) for c in f if c in scaler_features]
        X_model = X_scaled[:, model_feature_indices]
        #X_subscaled = feature_scaler.transform(X_sub)
        y_predscaled = m.predict(X_model)
        y_pred = target_scaler.inverse_transform(y_predscaled.reshape(-1, 1))
        y_pred = np.array(y_pred).reshape(-1)  # flatten in case shape is (n,1)
        return y_pred

    def predict_simple(m, f, df):
        # 1. Align features with the scaler first (scaler expects 8 features usually)
        scaler_features = list(feature_scaler.feature_names_in_) if hasattr(feature_scaler, "feature_names_in_") else f
        X_full = pd.DataFrame(0, index=df.index, columns=scaler_features)

        # 2. Fill the columns the model actually uses from filtered_df
        for col in f:
            if col in df.columns:
                X_full[col] = df[col]

        # 3. Scale the full feature set
        X_scaled = feature_scaler.transform(X_full)

        # 4. Filter only the features the model was trained on
        # This matches the indices of the model_features within the scaler_features
        model_feature_indices = [scaler_features.index(c) for c in f]
        X_model = X_scaled[:, model_feature_indices]

        # 5. Predict and Inverse Transform
        y_pred_scaled = m.predict(X_model)
        y_pred_unscaled = target_scaler.inverse_transform(y_pred_scaled.reshape(-1, 1)).ravel()
        return y_pred_unscaled

    print(f"Predicting using simple model: {modelname}")
    y_pred = predict_simple(gb_model, model_features, X_missing)


    #if method in ["stacking"]:
    #    meta_model = ensemble_model["meta_model"]
    #    test_preds = np.column_stack([safe_predict(m, f, X_missing) for m, f in base_models])
    #    y_pred = meta_model.predict(test_preds)

    #elif method in ["bagging"]:
    #    n_bags = ensemble_model.get("n_bags", 1)
    #    bag_preds = np.zeros((len(X_missing), len(base_models) * n_bags))
    #    idx = 0
    #    for m, features in base_models:
    #        for b in range(n_bags):
    #            bag_preds[:, idx] = safe_predict(m, features, X_missing)
    #            idx += 1
    #    y_pred = np.mean(bag_preds, axis=1)

    #elif method in ["weighted", "optimised", "multi_optimised"]:
    #    weights = np.array(ensemble_model.get("weights"))
    #    test_preds = np.column_stack([safe_predict(m, f, X_missing) for m, f in base_models])
    #    y_pred = np.dot(test_preds, weights)

    #else:
    #   raise ValueError("Unknown ensemble method")
    return y_pred


### **Replace CT NAN values with predicted CT values**

# --- STEP 3: WRAP IN HOM PHYSICS ---
def piml_predict_log_removal(k_pred,Clog, time):
    # Physics Calculation: Log10(N0/N) = (k * C * t^m) / 2.303
    m=2
    k_pred = np.asarray(k_pred).flatten()
    Clog = np.asarray(Clog).flatten()
    time = np.asarray(time).flatten()
    log_rem_pred = -(k_pred * Clog * (time**m)) / np.log10(10)
    return -log_rem_pred

def plot_feature_importance_logremoval(best_model, input_features, model_choice, X_train):
    """
    Plot feature importance and SHAP values for Random Forest, XGBoost,
    and ensemble models.

    Parameters:
    - best_model: trained model or list of trained models
    - input_features: list of feature names
    - model_choice: integer from 1 to 5
    - X: input DataFrame or array (used for SHAP)
    """
    results_dir ="Ensemble model 7 features - "
    os.makedirs(results_dir, exist_ok=True)
    def plot_importance_bar(importances, title):
        importance_df = pd.DataFrame({"Feature": input_features, "Importance": importances})
        importance_df = importance_df.sort_values(by="Importance", ascending=False)
        plt.figure(figsize=(10, 6))
        plt.barh(importance_df["Feature"], importance_df["Importance"], color='skyblue')
        plt.xlabel("Feature Importance Score")
        plt.ylabel("Features")
        plt.title(title)
        plt.gca().invert_yaxis()
        plt.savefig(os.path.join(results_dir, "Feauture importance.png"))
        plt.show()

    def plot_shap_summary(model, X_data, title):
        explainer = shap.Explainer(model, X_data)
        shap_values = explainer(X_data)
        shap.plots.beeswarm(shap_values, show=True, max_display=10)

    if model_choice == 1:  # Random Forest
        print("Random Forest - Feature Importance")
        plot_importance_bar(best_model.feature_importances_, "Random Forest - Feature Importance")
        print("Random Forest - SHAP Summary")
        plot_shap_summary(best_model, X_train, "Random Forest - SHAP Summary")

    elif model_choice == 2:  # XGBoost
        print("XGBoost - Feature Importance")
        plot_importance_bar(best_model.feature_importances_, "XGBoost - Feature Importance")
        print("XGBoost - SHAP Summary")
        plot_shap_summary(best_model, X_train, "XGBoost - SHAP Summary")

    elif model_choice == 3:  # GBoost
        print("GBoost - Feature Importance")
        plot_importance_bar(best_model.feature_importances_, "GBoost - Feature Importance")
        print("GBoost - SHAP Summary")
        plot_shap_summary(best_model, X_train, "GBoost - SHAP Summary")

    elif model_choice == 4:  # AdaBoost
        print("AdaBoost - Feature Importance")
        plot_importance_bar(best_model.feature_importances_, "AdaBoost - Feature Importance")
        print("AdaBoost - SHAP Summary")
        background = shap.sample(X_train, 100).values
        X_sample = shap.sample(X_train, 100).values
        explainer = shap.KernelExplainer(best_model.predict, background)

        shap_values = explainer.shap_values(X_sample)

        shap.summary_plot(shap_values, X_sample, feature_names=input_features)

    elif model_choice == 5:  # LGBoost
        print("LGBoost - Feature Importance")
        plot_importance_bar(best_model.feature_importances_, "LGBoost - Feature Importance")
        print("LGBoost - SHAP Summary")
        plot_shap_summary(best_model, X_train, "LGBoost - SHAP Summary")

    else:
        raise ValueError("Feature importance is only supported for model choices 1 to 5.")

def predict_logremoval_outlet(X, y, input_features, target_variable, models_info,log_removal_observed,Clog,time):
    """
    Main function to predict Bromate using user-selected model and inputs.
    """

    # Allow user to select a model
    print("\nSelect a model:")
    print("1. Random Forest")
    print("2. XGBoost")
    print("3. Gradient Boosting")
    print("4. AdaBoost")
    print("5. LightGBM")
    print("6. Physics-Informed MLP")
    print("7. Best 4 models weighted")
    try:
        model_choice = int(input("Enter the number corresponding to your choice:"))
    except ValueError:
        print("Invalid choice. Exiting.")
        return

    # Train-test split
    testsize = float(input("Set the testing size(e.g. 0.1 = 10% for testing):"))
    rdmstate = int(input("Set the random state:"))
    (X_train, X_test,
     y_train, y_test,
     Clog_train, Clog_test,
     time_train, time_test,
     log_removal_obs_train, log_removal_obs_test, # Added this log_removal_obs_test
     idx_train, idx_test) = train_test_split(
        X, y, Clog, time, log_removal_observed, range(len(X)),
        test_size=testsize, random_state=rdmstate
    )
    feature_scaler = StandardScaler()
    target_scaler = StandardScaler()

    X_train_scaled = feature_scaler.fit_transform(X_train)
    X_test_scaled = feature_scaler.transform(X_test)
    y_train_scaled = target_scaler.fit_transform(y_train.values.reshape(-1, 1))
    save_dir = "/content/drive/My Drive/Colab Notebooks/Ozonation model/"
    scaler_dir = input('Set the folder name:').strip()
    full_save_dir = os.path.join(save_dir, scaler_dir)
    os.makedirs(full_save_dir, exist_ok=True)
    model_name1=f"{model_choice}feature_scaler.pkl"
    model_name2=f"{model_choice}target_scaler.pkl"
    save_path1=os.path.join(full_save_dir, model_name1)
    save_path2=os.path.join(full_save_dir, model_name2)
    joblib.dump(feature_scaler, save_path1)
    joblib.dump(target_scaler, save_path2)
    X_train = pd.DataFrame(X_train, columns=input_features)
    X_test = pd.DataFrame(X_test, columns=input_features)
    #X_train_ps, X_test_ps, y_train_ps, y_test_ps,idx_train, idx_test = train_test_split(X, y,range(len(X)), test_size=testsize, random_state=42)# this is for the PINN approach

    # Model selection
    best_model = None

    if model_choice == 1:
        model_type = "RF"
        y_train_scaled = y_train_scaled.ravel() # Reshape y_train before fitting
        %run "/content/drive/My Drive/Colab Notebooks/Ozonation model/ozonemodel.ipynb"
        best_model = random_forest_model(X_train_scaled, y_train_scaled)
        y_pred = best_model.predict(X_test_scaled)
        y_pred = target_scaler.inverse_transform(y_pred.reshape(-1, 1))
        #y_test = target_scaler.inverse_transform(y_test.reshape(-1, 1))

    elif model_choice == 2:
        model_type = "XGB"
        y_train_scaled = y_train_scaled.ravel() # Reshape y_train before fitting
        %run "/content/drive/My Drive/Colab Notebooks/Ozonation model/ozonemodel.ipynb"
        best_model = xgboost_model(X_train_scaled, y_train_scaled)
        y_pred = best_model.predict(X_test_scaled)
        y_pred = target_scaler.inverse_transform(y_pred.reshape(-1, 1))
        #y_test = target_scaler.inverse_transform(y_test.reshape(-1, 1))

    elif model_choice == 3:
        model_type = "GB"
        y_train_scaled = y_train_scaled.ravel() # Reshape y_train before fitting
        %run "/content/drive/My Drive/Colab Notebooks/Ozonation model/ozonemodel.ipynb"
        best_model = gb_model(X_train_scaled, y_train_scaled)
        y_pred = best_model.predict(X_test_scaled)
        y_pred = target_scaler.inverse_transform(y_pred.reshape(-1, 1))
        #y_test = target_scaler.inverse_transform(y_test.reshape(-1, 1))

    elif model_choice == 4:
        model_type = "AdaB"
        y_train_scaled = y_train_scaled.ravel() # Reshape y_train before fitting
        %run "/content/drive/My Drive/Colab Notebooks/Ozonation model/ozonemodel.ipynb"
        best_model = adaboost_model(X_train_scaled, y_train_scaled)
        y_pred = best_model.predict(X_test_scaled)
        y_pred = target_scaler.inverse_transform(y_pred.reshape(-1, 1))
        #y_test = target_scaler.inverse_transform(y_test.reshape(-1, 1))

    elif model_choice == 5:
        model_type = "LGB"
        y_train_scaled = y_train_scaled.ravel() # Reshape y_train before fitting
        %run "/content/drive/My Drive/Colab Notebooks/Ozonation model/ozonemodel.ipynb"
        best_model = lgb_model(X_train_scaled, y_train_scaled)
        y_pred = best_model.predict(X_test)
        y_pred = target_scaler.inverse_transform(y_pred.reshape(-1, 1))
        #y_test = target_scaler.inverse_transform(y_test.reshape(-1, 1))

    elif model_choice == 6:
        model_type = "PINN - Physics"
        use_mlp=input("Use MLP correction or pure physics model? (y/n):")
        if use_mlp == "y":
            use_mlp = True
            model_type = "PINN - Physics + MLP"
        else:
            use_mlp = False
            model_type = "Physics only model"
        pinn, k_calibrated_val, F_calibrated_val,y_pred = physics_informed_bromate_mlp(X_train_ps, X_test_ps, y_train_ps, k_calibrated, F_calibrated,use_mlp)
        best_model = pinn
        #y_pred = y_pred.to_numpy()
        y_pred = y_pred.reshape(-1, 1)
        y_test = y_test.values.reshape(-1, 1)
        #pinn, Ko3.item(), Kuva.item()

    else:
        print("Invalid choice. Exiting.")
        return None

    # Calculate performance metrics
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)
    si = rmse/np.mean(y_test)
    kge = kling_gupta_efficiency(y_test, y_pred)

    print("\nPerformance for predicting inactivation rate k:")
    print(f"MAE: {mae:.4f}")
    print(f"MSE: {mse:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"Scatter Index: {si:.4f}")
    print(f"R²: {r2:.4f}")
    print(f"Kling Gupta Efficiency: {kge:.4f}")

    # Calculate Log removal using HOM model and performance metrics performance metrics
    log_rem_pred = piml_predict_log_removal(y_pred,Clog_test, time_test)
    log_removal_obs_test = - log_removal_obs_test
    mae1 = mean_absolute_error(log_removal_obs_test, log_rem_pred)
    mse1 = mean_squared_error(log_removal_obs_test, log_rem_pred)
    rmse1 = np.sqrt(mse1)
    r21 = r2_score(log_removal_obs_test, log_rem_pred)
    si1 = rmse/np.mean(log_removal_obs_test)
    kge1 = kling_gupta_efficiency(log_removal_obs_test, log_rem_pred)

    print("\nPerformance for predicting log removal of coliform bacteria:")
    print(f"MAE: {mae1:.4f}")
    print(f"MSE: {mse1:.4f}")
    print(f"RMSE: {rmse1:.4f}")
    print(f"Scatter Index: {si1:.4f}")
    print(f"R²: {r21:.4f}")
    print(f"Kling Gupta Efficiency: {kge1:.4f}")

    # Create results directory

    results_dir = f"{model_type} {X.shape[1]} features"
    os.makedirs(results_dir, exist_ok=True)

    # Save performance metrics to a txt file
    metrics_file = os.path.join(results_dir, "performance_metrics.txt")
    with open(metrics_file, "w") as f:
        f.write("Performance for predicting inactivation rate k:\n")
        f.write(f"MAE: {mae:.4f}\n")
        f.write(f"MSE: {mse:.4f}\n")
        f.write(f"RMSE: {rmse:.4f}\n")
        f.write(f"Scatter Index: {si:.4f}\n")
        f.write(f"R²: {r2:.4f}\n")
        f.write(f"Kling Gupta Efficiency: {kge:.4f}\n")
        f.write("\n")
        f.write("Performance for predicting log removal of coliform bacteria:\n")
        f.write(f"MAE: {mae1:.4f}\n")
        f.write(f"MSE: {mse1:.4f}\n")
        f.write(f"RMSE: {rmse1:.4f}\n")
        f.write(f"Scatter Index: {si1:.4f}\n")
        f.write(f"R²: {r21:.4f}\n")
        f.write(f"Kling Gupta Efficiency: {kge1:.4f}\n")

    # Save input features if available
    try:
        feature_file = os.path.join(results_dir, f"{X.shape[1]} used_features.txt")
        with open(feature_file, "w") as f:
            f.write(f"Number of Features: {X.shape[1]}\n")
            f.write("Input Features Used:\n")
            f.write("\n".join(X_test.columns))
    except:
        pass  # Skip if X_test doesn't have .columns

    # Save predicted vs observed plot
    plt.figure(figsize=(6, 6))
    plt.scatter(log_removal_obs_test, log_rem_pred, color='blue', alpha=0.7, label='Predictions')
    plt.plot([log_removal_obs_test.min(), log_removal_obs_test.max()], [log_removal_obs_test.min(), log_removal_obs_test.max()], 'r--', label='1:1 Line')
    plt.xlabel("Observed Log Removal")
    plt.ylabel("Predicted Log Removal")
    plt.title("Predicted vs Observed Log Removal")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(results_dir, "predicted_vs_observed.png"))
    plt.show()

    # Plot predicted vs observed data
    #plt.scatter(y_test, y_pred, color='blue')
    #plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], color='red', linestyle='--')
    #plt.xlabel("Observed CT")
    #plt.ylabel("Predicted CT")
    #plt.title("Predicted vs Observed CT")
    #plt.show()

    # Plot feature importance if model is Random Forest or XGBoost
    if model_choice in [1, 2, 3, 4, 5]:
        plot_feature_importance_logremoval(best_model, input_features,model_choice,X_train)

    return best_model, mae, mse, rmse, r2, si,y_test, y_pred
